SPDX-FileCopyrightText: Copyright (c) 2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved. <br>
SPDX-License-Identifier: Apache-2.0 <br>

Licensed under the Apache License, Version 2.0 (the "License"); <br>
you may not use this file except in compliance with the License. <br>
You may obtain a copy of the License at <br>

http://www.apache.org/licenses/LICENSE-2.0  

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, <br>
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. <br>
See the License for the specific language governing permissions and <br>
limitations under the License. 

In [8]:
import time

import torch
from torch import nn

from warpkan.torch_ext.ukanlinear import UKANLinear
from warpkan.torch_ext.kan import KAN

import warp as wp

# MLP vs UKAN

In [10]:
wp.init()
in_dim = out_dim = 2
mlp_hidden = 64
ukan = nn.Sequential(UKANLinear(in_dim, 10, 1, 0.2, 16, has_mlp=False), UKANLinear(10, out_dim, 1, 0.2, 16, has_mlp=False)).cuda()
mlp = nn.Sequential(nn.Linear(in_dim, mlp_hidden), nn.ReLU(), nn.Linear(mlp_hidden, mlp_hidden), nn.ReLU(), nn.Linear(mlp_hidden, in_dim)).cuda()
print("UKAN params: ", sum(p.numel() for p in ukan.parameters()))
print("MLP params: ", sum(p.numel() for p in mlp.parameters()))


UKAN params:  1696
MLP params:  4482


In [11]:
x = torch.randn(1024, in_dim).cuda()
x_val = torch.randn(1024, in_dim).cuda()
freq = torch.linspace(1, 7, in_dim).cuda()
target = torch.sin( x * freq)

target_val = torch.sin( x_val * freq)

In [12]:
for model in [ukan, mlp]:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    print(model)
    for epoch in range(10000):
        optimizer.zero_grad()
        y = model(x)
        loss = torch.mean((target-y)**2)
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            if epoch == 0:
                print(f"{'epoch':5} | {'train_loss':12} | {'val_loss':12}")    
            print(f"{epoch:5} | {loss.item():0.10f} | {torch.mean((target_val- model(x_val))**2).item():0.10f}")

Sequential(
  (0): UKANLinear(
    (base_activation): SiLU()
    (coeff_gen): Sequential(
      (0): Linear(in_features=32, out_features=16, bias=True)
      (1): SiLU()
      (2): Linear(in_features=16, out_features=20, bias=True)
    )
    (ubspline): UBSpline()
    (embed): SinusoidalPositionalEncoding()
    (embed_loc): Embedding(2, 16)
  )
  (1): UKANLinear(
    (base_activation): SiLU()
    (coeff_gen): Sequential(
      (0): Linear(in_features=32, out_features=16, bias=True)
      (1): SiLU()
      (2): Linear(in_features=16, out_features=4, bias=True)
    )
    (ubspline): UBSpline()
    (embed): SinusoidalPositionalEncoding()
    (embed_loc): Embedding(10, 16)
  )
)
epoch | train_loss   | val_loss    
    0 | 0.4568920135 | 0.4558911324
  500 | 0.2584937811 | 0.2641223669
 1000 | 0.2145970166 | 0.2312691510
 1500 | 0.0693099946 | 0.0859852657
 2000 | 0.0344783626 | 0.0446015783
 2500 | 0.0214377381 | 0.0299978405
 3000 | 0.0113540171 | 0.0166842490
 3500 | 0.0056180758 | 0.010

## warp KAN vs efficient public KAN

* Cost is proportional to order not grid_dim O(grid_dim * rest) -> O(order * rest)
* Memory efficient 


In [14]:
x = torch.randn(2**20, in_dim).cuda()


In [ ]:
from efficient_kan import KAN as PKAN
public_kan = PKAN([in_dim, 10, in_dim], grid_size=grid_num, grid_range=[-5, 5]).cuda()
for _ in range(100):
    public_kan(x)
torch.cuda.synchronize()


## Param Size vs KAN:

* cost does not depend on warp KAN parameters directly 

In [15]:
for grid_num in [32, 64, 128, 256, 512, 1024]:
    kan = KAN([in_dim, 10, in_dim], grid_num, 3, grid_range=[-5, 5]).cuda()
    for _ in range(100):
        kan(x)
    torch.cuda.synchronize()
    s = time.perf_counter()
    for _ in range(100):
        kan(x)
    torch.cuda.synchronize()
    e = time.perf_counter()
    print("KAN params: ", sum(p.numel() for p in kan.parameters()), ", dt: ", 10*(e - s))
    

Module bspline_fwd_3_2_10_41_torchfloat32 17479ee load on device 'cuda:0' took 308.81 ms  (compiled)
Module bspline_fwd_3_10_2_41_torchfloat32 8aa9a34 load on device 'cuda:0' took 730.17 ms  (compiled)
KAN params:  1520 , dt:  1.6838992503471673
Module bspline_fwd_3_2_10_73_torchfloat32 9dd8c2c load on device 'cuda:0' took 277.41 ms  (compiled)
Module bspline_fwd_3_10_2_73_torchfloat32 78b6bec load on device 'cuda:0' took 715.79 ms  (compiled)
KAN params:  2800 , dt:  1.6960617899894714
Module bspline_fwd_3_2_10_137_torchfloat32 f6959c7 load on device 'cuda:0' took 311.80 ms  (compiled)
Module bspline_fwd_3_10_2_137_torchfloat32 6497bef load on device 'cuda:0' took 713.93 ms  (compiled)
KAN params:  5360 , dt:  1.8282037205062807
Module bspline_fwd_3_2_10_265_torchfloat32 1a7132f load on device 'cuda:0' took 280.08 ms  (compiled)
Module bspline_fwd_3_10_2_265_torchfloat32 edbd1c6 load on device 'cuda:0' took 755.84 ms  (compiled)
KAN params:  10480 , dt:  1.8650472187437117
Module bspl

In [16]:
in_dim = out_dim = 5
for grid_num in [32, 64, 128, 256, 512, 1024]:
    public_kan = PKAN([in_dim, 10, in_dim], grid_size=grid_num, grid_range=[-5, 5]).cuda()
    try:
        for _ in range(100):
            public_kan(x)
        torch.cuda.synchronize()
        s = time.perf_counter()
        for _ in range(100):
            public_kan(x)
        torch.cuda.synchronize()
        e = time.perf_counter()
        print("Public KAN params: ", sum(p.numel() for p in kan.parameters()), ", dt: ", 10*(e - s))
    except:
        print("Public KAN params: ", sum(p.numel() for p in kan.parameters()), " failed!")
        break
    

NameError: name 'PKAN' is not defined